In [0]:
# Databricks notebook source
from typing import Any

# COMMAND ----------

import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

# COMMAND ----------

dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:

# COMMAND ----------

def load_curated_auction_real_estate_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "AuctionRealEstateId String, OriginSourceSystem String, SourceSystem String, IngestionTimestamp timestamp, BatchId String,"
        #     "AuctionRealEstateKey String, AuctionRealEstateCode String, AuctionCode String, AuctionObjKey string, RealEstateGroupCode int,"
        #     "DebtorPartyKey String, OriginDebtorTaxRegistrationNumber String, OriginDebtorDescription String, AuctionDate Date, AuctionPublishedDate date, AuctionPostedDate date,"
        #     "PartyKey string, RealEstateKey string, RealEstateOwnershipKey string, CaseCode int,"
        #     "AuctionFirstOfferPrice String, SeizureReportNumber String, SeizureReportYear int, HastenerCode String, Hastener String, HastenerTaxRegistrationNumber String,"
        #     "AuctionFinalPrice String, AuctionStatusCode String, AuctionStatus String, AuctioneerName String, AuctioneerEmail String, AuctioneerPhone String, TitleGR String,"
        #     "Area String, District String, Municipality String, Prefecture String, GeoCoordinatesX String, GeoCoordinatesY String, GeoLevel String, Address String,"
        #     "PostalCode int, Category String, SubCategory String, LandeaUrl String, FloorRawValue String, ConstructionYear int, SurfaceSqm float, Tags String,"
        #     "TagValues String, ActiveFlag boolean, TNMergeCode String, LandRegistration String, BidderCode String, DateEntered String, EnteredBy String, AuctionOwnershipTypeCode int,"
        #     "AuctionOwnershipType String, FirstBidder String, LawFirm String,FileDate date,FileName string, BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.AuctionRealEstate"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qca",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== FS_LEGAL_PROCESS ==================

        FS_LEGAL_PROCESS_extra_filters = None

        if FS_LEGAL_PROCESS_extra_filters is not None:
            FS_LEGAL_PROCESS_extra_filters.append(date_filter)
            FS_LEGAL_PROCESS_final_filters = " AND ".join(
                FS_LEGAL_PROCESS_extra_filters
            )
        else:
            FS_LEGAL_PROCESS_final_filters = date_filter

        FS_LEGAL_PROCESS_selected_fields = [
            "LPS_CODE",
            "AUST_ID",
            "WKFV_AUCTION_ID",
            "FSLP_OPEN_BID_AMT_LCY",
            "LPINIT_ID",
            "EXTC_ID",
            "businessdate",
        ]

        FS_LEGAL_PROCESS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_LEGAL_PROCESS",
            FS_LEGAL_PROCESS_final_filters,
            FS_LEGAL_PROCESS_selected_fields,
        )

        # ================== FSAD_LEGAL_PROCESS_CASES ==================

        FSAD_LEGAL_PROCESS_CASES_extra_filters = None

        if FSAD_LEGAL_PROCESS_CASES_extra_filters is not None:
            FSAD_LEGAL_PROCESS_CASES_extra_filters.append(date_filter)
            FSAD_LEGAL_PROCESS_CASES_final_filters = " AND ".join(
                FSAD_LEGAL_PROCESS_CASES_extra_filters
            )
        else:
            FSAD_LEGAL_PROCESS_CASES_final_filters = date_filter

        FSAD_LEGAL_PROCESS_CASES_selected_fields = [
            "LPS_CODE",
            "CASE_CODE",
            "businessdate",
        ]

        FSAD_LEGAL_PROCESS_CASES = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FSAD_LEGAL_PROCESS_CASES",
            FSAD_LEGAL_PROCESS_CASES_final_filters,
            FSAD_LEGAL_PROCESS_CASES_selected_fields,
        )

        # ================== FS_CASES_SNAP ==================

        FS_CASES_SNAP_extra_filters = None

        if FS_CASES_SNAP_extra_filters is not None:
            FS_CASES_SNAP_extra_filters.append(date_filter)
            FS_CASES_SNAP_final_filters = " AND ".join(FS_CASES_SNAP_extra_filters)
        else:
            FS_CASES_SNAP_final_filters = date_filter

        FS_CASES_SNAP_selected_fields = [
            "CASE_CODE",
            "CUST_CODE",
            "businessdate",
        ]

        FS_CASES_SNAP = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FS_CASES_SNAP",
            FS_CASES_SNAP_final_filters,
            FS_CASES_SNAP_selected_fields,
        )

        # ================== DIM_CASES_STATIC ==================

        DIM_CASES_STATIC_extra_filters = None

        if DIM_CASES_STATIC_extra_filters is not None:
            DIM_CASES_STATIC_extra_filters.append(date_filter)
            DIM_CASES_STATIC_final_filters = " AND ".join(
                DIM_CASES_STATIC_extra_filters
            )
        else:
            DIM_CASES_STATIC_final_filters = date_filter

        DIM_CASES_STATIC_selected_fields = [
            "CASE_CODE",
            "businessdate",
        ]

        DIM_CASES_STATIC = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_DIM_CASES_STATIC",
            DIM_CASES_STATIC_final_filters,
            DIM_CASES_STATIC_selected_fields,
        )

        # ================== FSAD_LEGAL_PROCESS_ITEMS ==================

        FSAD_LEGAL_PROCESS_ITEMS_extra_filters = None

        if FSAD_LEGAL_PROCESS_ITEMS_extra_filters is not None:
            FSAD_LEGAL_PROCESS_ITEMS_extra_filters.append(date_filter)
            FSAD_LEGAL_PROCESS_ITEMS_final_filters = " AND ".join(
                FSAD_LEGAL_PROCESS_ITEMS_extra_filters
            )
        else:
            FSAD_LEGAL_PROCESS_ITEMS_final_filters = date_filter

        FSAD_LEGAL_PROCESS_ITEMS_selected_fields = [
            "LPS_CODE",
            "PRO_CODE",
            "CUST_CODE",
            "businessdate",
        ]

        FSAD_LEGAL_PROCESS_ITEMS = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FSAD_LEGAL_PROCESS_ITEMS",
            FSAD_LEGAL_PROCESS_ITEMS_final_filters,
            FSAD_LEGAL_PROCESS_ITEMS_selected_fields,
        )

        FSAD_LEGAL_PROCESS_ITEMS = FSAD_LEGAL_PROCESS_ITEMS.distinct()
        # ================== FSAD_PROPERTIES_REAL_ESTATES ==================

        FSAD_PROPERTIES_REAL_ESTATES_extra_filters = None

        if FSAD_PROPERTIES_REAL_ESTATES_extra_filters is not None:
            FSAD_PROPERTIES_REAL_ESTATES_extra_filters.append(date_filter)
            FSAD_PROPERTIES_REAL_ESTATES_final_filters = " AND ".join(
                FSAD_PROPERTIES_REAL_ESTATES_extra_filters
            )
        else:
            FSAD_PROPERTIES_REAL_ESTATES_final_filters = date_filter

        FSAD_PROPERTIES_REAL_ESTATES_selected_fields = [
            "PRO_CODE",
            "PRO_ID",
            "CUST_CODE",
            "REST_TOPONYM",
            "REST_PREFECTURE",
            "REST_CATEGORY_CODE",
            "PRO_ASSET_TYPE_CODE",
            "REST_CODE",
            "businessdate",
        ]

        FSAD_PROPERTIES_REAL_ESTATES = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_FSAD_PROPERTIES_REAL_ESTATES",
            FSAD_PROPERTIES_REAL_ESTATES_final_filters,
            FSAD_PROPERTIES_REAL_ESTATES_selected_fields,
        )

        # Join Conditions

        FS_LEGAL_PROCESS_FSAD_LEGAL_PROCESS_CASES_condition = (
            FS_LEGAL_PROCESS["LPS_CODE"] == FSAD_LEGAL_PROCESS_CASES["LPS_CODE"]
        )

        FS_CASES_SNAP_FSAD_LEGAL_PROCESS_CASES_condition = (
            FS_CASES_SNAP["CASE_CODE"] == FSAD_LEGAL_PROCESS_CASES["CASE_CODE"]
        )

        FS_CASES_SNAP_DIM_CASES_STATIC_condition = (
            FS_CASES_SNAP["CASE_CODE"] == DIM_CASES_STATIC["CASE_CODE"]
        )

        FS_LEGAL_PROCESS_FSAD_LEGAL_PROCESS_ITEMS_condition = (
            FS_LEGAL_PROCESS["LPS_CODE"] == FSAD_LEGAL_PROCESS_ITEMS["LPS_CODE"]
        )

        FSAD_PROPERTIES_REAL_ESTATES_FSAD_LEGAL_PROCESS_ITEMS_condition = (
            FSAD_PROPERTIES_REAL_ESTATES["PRO_CODE"]
            == FSAD_LEGAL_PROCESS_ITEMS["PRO_CODE"]
        ) & (
            FSAD_PROPERTIES_REAL_ESTATES["CUST_CODE"]
            == FSAD_LEGAL_PROCESS_ITEMS["CUST_CODE"]
        )

        # Joins
        target_table_qualco_ab = (
            FS_LEGAL_PROCESS.join(
                FSAD_LEGAL_PROCESS_CASES,
                FS_LEGAL_PROCESS_FSAD_LEGAL_PROCESS_CASES_condition,
                "inner",
            )
            .join(
                FS_CASES_SNAP, FS_CASES_SNAP_FSAD_LEGAL_PROCESS_CASES_condition, "inner"
            )
            .join(DIM_CASES_STATIC, FS_CASES_SNAP_DIM_CASES_STATIC_condition, "inner")
            .join(
                FSAD_LEGAL_PROCESS_ITEMS,
                FS_LEGAL_PROCESS_FSAD_LEGAL_PROCESS_ITEMS_condition,
                "inner",
            )
            .join(
                FSAD_PROPERTIES_REAL_ESTATES,
                FSAD_PROPERTIES_REAL_ESTATES_FSAD_LEGAL_PROCESS_ITEMS_condition,
                "inner",
            )
        )

        # HashKeys
        target_table_qualco_ab_AuctionRealEstateId_primary_key_fields = [
            FS_LEGAL_PROCESS.LPS_CODE,
            FSAD_LEGAL_PROCESS_ITEMS.PRO_CODE,
            FS_CASES_SNAP.CASE_CODE,
            lit("qca"),
        ]
        target_table_qualco_ab_RealEstateId_primary_key_fields = [
            FSAD_PROPERTIES_REAL_ESTATES.REST_CODE,
            lit("qca"),
        ]
        target_table_qualco_ab_PartyId_primary_key_fields = [
            FS_CASES_SNAP.CUST_CODE,
            lit("qca"),
        ]

        target_table_qualco_ab = (
            target_table_qualco_ab.withColumn(
                "AuctionRealEstateId",
                sha2(
                    concat_ws(
                        ";",
                        *target_table_qualco_ab_AuctionRealEstateId_primary_key_fields,
                    ),
                    256,
                ),
            )
            .withColumn(
                "RealEstateId",
                sha2(
                    concat_ws(
                        ";", *target_table_qualco_ab_RealEstateId_primary_key_fields
                    ),
                    256,
                ),
            )
            .withColumn(
                "PartyId",
                sha2(
                    concat_ws(";", *target_table_qualco_ab_PartyId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "AuctionDate",
                concat(
                    substring(FS_LEGAL_PROCESS.WKFV_AUCTION_ID, 1, 4),
                    lit("-"),
                    substring(FS_LEGAL_PROCESS.WKFV_AUCTION_ID, 5, 2),
                    lit("-"),
                    substring(FS_LEGAL_PROCESS.WKFV_AUCTION_ID, 7, 2),
                ),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qualco_ab.select(
            target_table_qualco_ab.AuctionRealEstateId.alias("AuctionRealEstateId"),
            lit(None).alias("OriginSourceSystem"),
            lit("qca").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            target_table_qualco_ab.AuctionRealEstateId.cast("string").alias(
                "AuctionRealEstateKey"
            ),
            lit(None).alias("AuctionRealEstateCode"),
            lit(None).alias("AuctionCode"),
            lit(None).alias("AuctionObjKey"),
            lit(None).alias("RealEstateGroupCode"),
            # target_table_qualco_ab.RealEstateId.alias("RealEstateId"),
            target_table_qualco_ab.PartyId.cast("string").alias("DebtorPartyKey"),
            lit(None).alias("OriginDebtorTaxRegistrationNumber"),
            lit(None).alias("OriginDebtorDescription"),
            target_table_qualco_ab.AuctionDate.cast("Date").alias(
                "AuctionDate"
            ),  # check if need to change to date
            lit(None).alias("AuctionFinalPrice"),
            lit(None).alias("AuctionPublishedDate"),
            lit(None).alias("AuctionPostedDate"),
            FS_CASES_SNAP.CUST_CODE.cast("String").alias("PartyKey"),
            target_table_qualco_ab.REST_CODE.cast("String").alias("RealEstateKey"),
            target_table_qualco_ab.PRO_ID.cast("String").alias(
                "RealEstateOwnershipKey"
            ),
            FS_CASES_SNAP.CASE_CODE.alias("CaseCode"),
            target_table_qualco_ab.FSLP_OPEN_BID_AMT_LCY.cast("string").alias(
                "AuctionFirstOfferPrice"
            ),  # check
            lit(None).alias("SeizureReportNumber"),
            lit(None).alias("SeizureReportYear"),
            lit(None).alias("HastenerCode"),
            lit(None).alias("Hastener"),
            lit(None).alias("HastenerTaxRegistrationNumber"),
            FS_LEGAL_PROCESS.AUST_ID.cast("String").alias("AuctionStatusCode"),
            lit(None).alias("AuctionStatus"),
            lit(None).alias("AuctioneerName"),
            lit(None).alias("AuctioneerEmail"),
            lit(None).alias("AuctioneerPhone"),
            lit(None).alias("TitleGR"),
            lit(None).alias("Area"),
            lit(None).alias("District"),
            FSAD_PROPERTIES_REAL_ESTATES.REST_TOPONYM.cast("String").alias(
                "Municipality"
            ),
            FSAD_PROPERTIES_REAL_ESTATES.REST_PREFECTURE.cast("String").alias(
                "Prefecture"
            ),
            lit(None).alias("GeoCoordinatesX"),
            lit(None).alias("GeoCoordinatesY"),
            lit(None).alias("GeoLevel"),
            lit(None).alias("Address"),
            lit(None).alias("PostalCode"),
            FSAD_PROPERTIES_REAL_ESTATES.REST_CATEGORY_CODE.cast("String").alias(
                "Category"
            ),
            lit(None).alias("SubCategory"),
            lit(None).alias("LandeaUrl"),
            lit(None).alias("FloorRawValue"),
            lit(None).alias("ConstructionYear"),
            lit(None).alias("SurfaceSqm"),
            lit(None).alias("Tags"),
            lit(None).alias("TagValues"),
            lit(None).alias("ActiveFlag"),
            lit(None).alias("TNMergeCode"),
            lit(None).alias("LandRegistration"),
            lit(None).alias("BidderCode"),
            lit(None).alias("DateEntered"),
            lit(None).alias("EnteredBy"),
            FSAD_PROPERTIES_REAL_ESTATES.PRO_ASSET_TYPE_CODE.cast("int").alias(
                "AuctionOwnershipTypeCode"
            ),
            lit(None).alias("AuctionOwnershipType"),
            FS_LEGAL_PROCESS.LPINIT_ID.alias("FirstBidder"),
            FS_LEGAL_PROCESS.EXTC_ID.cast("string").alias("LawFirm"),
            lit(None).alias("FileDate"),
            lit(None).alias("FileName"),
            FS_LEGAL_PROCESS.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:

# COMMAND ----------

load_curated_auction_real_estate_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)